# Diffusion for minority-class augmentation (Scania APS, PCA features)

This notebook implements a **DDPM-style diffusion model** for **tabular data** (here: PCA vectors).
The goal is to generate **synthetic minority-class samples** (APS failures) to reduce class imbalance.

---

## What problem are we solving?

In the APS dataset, failures are rare (minority class). Training directly on the raw distribution often leads to models that:
- look good on accuracy,
- but miss real failures (high false negatives).

Diffusion augmentation gives us a way to *learn the minority distribution* and sample additional realistic failures.

---

## Inputs

- `../../data/processed/pca_aps_mean_failure_train_set.csv`
- `../../data/processed/pca_aps_mean_failure_test_set.csv`

These should contain:
- `pc_0 ... pc_10`
- `class` (0 = normal, 1 = failure)

---

## Outputs

This notebook writes three artifacts (with a timestamp run id):

- `diffusion_synth_failure_pca_<RUN_ID>.csv`  
  Synthetic failure samples in PCA space.

- `pca_aps_mean_failure_train_set_diffusion_augmented_<RUN_ID>.csv`  
  Augmented training set (original train + synthetic failures).

- `diffusion_denoiser_pca_<RUN_ID>.pth`  
  Saved PyTorch checkpoint of the trained denoiser + normalization stats.

---

## How to use this notebook

1. Run all cells top-to-bottom.
2. Use the augmented training CSV in the **evaluation notebook** to test whether augmentation reduces cost / improves recall.


In [ ]:
# --- Imports ---
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# --- Reproducibility + run id (used in output filenames) ---
np.random.seed(42)
torch.manual_seed(42)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
print("RUN_ID:", RUN_ID)


## 1) Load PCA train/test data

We load the PCA datasets and extract:
- the PCA feature columns (`pc_0 ... pc_10`)
- the label column `class`

We then split the training set into:
- minority failures (`class == 1`)
- majority normal samples (`class == 0`)


In [ ]:
train_path = Path("../../data/processed/pca_aps_mean_failure_train_set.csv")
test_path  = Path("../../data/processed/pca_aps_mean_failure_test_set.csv")

# If your files are inside ScaniaDataset/, uncomment:
# train_path = Path("ScaniaDataset/../../data/processed/pca_aps_mean_failure_train_set.csv")
# test_path  = Path("ScaniaDataset/../../data/processed/pca_aps_mean_failure_test_set.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

pc_cols = [c for c in train_df.columns if c.startswith("pc_")]
X_train = train_df[pc_cols].values.astype(np.float32)
y_train = train_df["class"].values.astype(np.int64)

X_min = X_train[y_train == 1]
X_maj = X_train[y_train == 0]

print("Train:", train_df.shape, " Test:", test_df.shape)
print("Minority:", X_min.shape[0], " Majority:", X_maj.shape[0])


## 2) Standardize failure-only data

Diffusion training is easier when the input space is roughly standardized.

Here we compute the mean and standard deviation **on failure samples only** and standardize failures:

`X_min_std = (X_min - mean_failure) / std_failure`

We save these stats into the model checkpoint so we can invert the transform at sampling time.


In [ ]:
min_mean = X_min.mean(axis=0, keepdims=True)
min_std  = X_min.std(axis=0, keepdims=True) + 1e-6
X_min_std = (X_min - min_mean) / min_std


## 3) Define DDPM diffusion + denoiser MLP

### Diffusion intuition + the key math (DDPM-style)

Diffusion models learn to generate data by reversing a gradual noising process.

#### Forward process (add noise)
Starting from a real data vector **x0**, we add Gaussian noise over **T** steps:

- Define a noise schedule: `beta_t` for `t = 1..T`
- `alpha_t = 1 - beta_t`
- `alpha_bar_t = Π_{s=1..t} alpha_s`

Then we can sample a noised version `x_t` in *one step*:

`x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps`, where `eps ~ N(0, I)`.

#### Reverse process (denoise)
We train a neural network `eps_theta(x_t, t)` to predict the noise `eps` that was added.

#### Training objective (noise prediction)
A common DDPM loss is:

`L = E[ || eps - eps_theta(x_t, t) ||^2 ]`

This is simple and stable: it is just mean-squared error on the noise.

In this notebook we use:
- a linear beta schedule
- a small MLP denoiser
- failure-only PCA vectors as the training distribution


In [ ]:
device = torch.device("cpu")
torch.set_num_threads(2)

T = 50
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def extract(a: torch.Tensor, t: torch.Tensor, x_shape):
    out = a.gather(0, t).float()
    return out.view(-1, *([1] * (len(x_shape) - 1)))

def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    sqrt_ab = torch.sqrt(extract(alpha_bars, t, x0.shape))
    sqrt_om = torch.sqrt(1.0 - extract(alpha_bars, t, x0.shape))
    return sqrt_ab * x0 + sqrt_om * noise

class Denoiser(nn.Module):
    def __init__(self, d: int, T: int, emb_dim: int = 32, hidden: int = 128):
        super().__init__()
        self.emb = nn.Embedding(T, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(d + emb_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, d),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        te = self.emb(t)
        return self.net(torch.cat([x, te], dim=1))

d = len(pc_cols)
model = Denoiser(d=d, T=T).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


## 4) Train on failures only

We train the denoiser MLP using the DDPM noise-prediction objective.

**Inputs:** standardized failure vectors  
**Output:** a trained `model` that predicts noise `eps` from `(x_t, t)`


In [ ]:
dataset = TensorDataset(torch.from_numpy(X_min_std))
loader = DataLoader(dataset, batch_size=256, shuffle=True, drop_last=True)

epochs = 500
model.train()

for ep in range(1, epochs + 1):
    running, n = 0.0, 0
    for (x0,) in loader:
        x0 = x0.to(device)
        b = x0.size(0)
        t = torch.randint(0, T, (b,), device=device, dtype=torch.long)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, noise)
        pred = model(xt, t)
        loss = loss_fn(pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()

        running += loss.item() * b
        n += b

    if ep % 100 == 0:
        print(f"Epoch {ep:4d}/{epochs}  loss={running/n:.4f}")


## 5) Sample synthetic failures

We start from pure Gaussian noise and iteratively apply the learned reverse steps from `t = T-1 ... 0`.

Finally we un-standardize the generated samples back into the PCA space:

`X_synth = X_synth_std * std_failure + mean_failure`


In [ ]:
@torch.no_grad()
def p_sample_loop(model: nn.Module, n_samples: int, batch_size: int = 2048) -> np.ndarray:
    model.eval()
    out = []
    alpha_bars_prev = torch.cat([torch.tensor([1.0], device=device), alpha_bars[:-1]], dim=0)

    for start in range(0, n_samples, batch_size):
        bs = min(batch_size, n_samples - start)
        x = torch.randn(bs, d, device=device)  # x_T

        for t_i in reversed(range(T)):
            t = torch.full((bs,), t_i, device=device, dtype=torch.long)

            beta_t = betas[t_i]
            alpha_t = alphas[t_i]
            ab_t = alpha_bars[t_i]
            ab_prev = alpha_bars_prev[t_i]

            eps = model(x, t)
            x0_pred = (x - torch.sqrt(1.0 - ab_t) * eps) / torch.sqrt(ab_t)

            coef1 = torch.sqrt(ab_prev) * beta_t / (1.0 - ab_t)
            coef2 = torch.sqrt(alpha_t) * (1.0 - ab_prev) / (1.0 - ab_t)
            mean = coef1 * x0_pred + coef2 * x
            var = beta_t * (1.0 - ab_prev) / (1.0 - ab_t)

            if t_i > 0:
                x = mean + torch.sqrt(var) * torch.randn_like(x)
            else:
                x = mean

        out.append(x.cpu().numpy())

    return np.vstack(out)

n_synth = int(X_maj.shape[0] - X_min.shape[0])
X_synth_std = p_sample_loop(model, n_samples=n_synth)
X_synth = X_synth_std * min_std + min_mean


## 6) Save synthetic + augmented CSVs

We write:
1) a CSV containing only synthetic failures, and  
2) an augmented training CSV that concatenates the original training set with the synthetic failures.

**Tip:** keep the test set untouched. Augmentation is applied to training only.


In [ ]:
synth_df = pd.DataFrame(X_synth, columns=pc_cols)
synth_df["class"] = 1

aug_df = pd.concat([train_df, synth_df], ignore_index=True)

out_synth = Path(f"diffusion_synth_failure_pca_{RUN_ID}.csv")
out_aug   = Path(f"pca_aps_mean_failure_train_set_diffusion_augmented_{RUN_ID}.csv")
model_ckpt = Path(f"diffusion_denoiser_pca_{RUN_ID}.pth")

synth_df.to_csv(out_synth, index=False)
aug_df.to_csv(out_aug, index=False)

torch.save({
    "state_dict": model.state_dict(),
    "min_mean": min_mean,
    "min_std": min_std,
    "T": T,
    "beta_start": float(betas[0].cpu()),
    "beta_end": float(betas[-1].cpu()),
}, model_ckpt)

print("Saved:", out_synth, out_aug, model_ckpt)
print("Augmented class counts:\n", aug_df["class"].value_counts())